### Cell 1 — Imports and Config

In [16]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, year, to_date, expr, trim, lower, when
from pyspark.sql.types import DoubleType, TimestampType, DateType
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

# ── Lakehouse routing ─────────────────────────────────────────────────────────
# Pre-silver staging tables live in the Silver lakehouse (not Bronze).
# This ensures spark.table() resolves them via the default catalog here
# without cross-lakehouse references — which caused TABLE_OR_VIEW_NOT_FOUND.
SILVER_CATALOG      = "lh_Silver_StratusCoreTelecoms"
BRONZE_CATALOG      = "lh_Bronze_StratusCoreTelecoms"
PRE_SILVER_TABLE    = f"{SILVER_CATALOG}.dbo.pre_silver_stratus_availability"
BRONZE_OUTAGE_TABLE = f"{BRONZE_CATALOG}.dbo.bronze_stratus_outage"

BRONZE_PATH = "abfss://Swift@onelake.dfs.fabric.microsoft.com/lh_Bronze_StratusCoreTelecoms.Lakehouse"

RUN_TS = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
print(f"\u2705 Silver config loaded \u2014 Run: {RUN_TS}")
print(f"   Pre-silver source : {PRE_SILVER_TABLE}")
print(f"   Outage source     : {BRONZE_OUTAGE_TABLE}")


StatementMeta(, 7f698acc-f2ee-4717-bc6b-d3383dfadf0e, 18, Finished, Available, Finished, False)

✅ Silver config loaded — Run: 2026-04-18 11:22:04
   Pre-silver source : lh_Silver_StratusCoreTelecoms.dbo.pre_silver_stratus_availability
   Outage source     : lh_Bronze_StratusCoreTelecoms.dbo.bronze_stratus_outage


### Cell 2 — Helper: Replace Null-Like String Values

In [17]:
def replace_null_strings(sdf, columns=None):
    """
    Replace common null-like string artifacts produced when pandas serialises
    missing values to string: 'nan', 'None', 'NaT', 'null', '<NA>', empty string.

    Parameters
    ----------
    sdf     : Spark DataFrame
    columns : list of column names to clean (default: all string columns)

    Returns
    -------
    Spark DataFrame with null-like strings replaced with actual None
    """
    NULL_STRINGS = {"nan", "none", "nat", "null", "<na>", ""}

    if columns is None:
        from pyspark.sql.types import StringType
        columns = [f.name for f in sdf.schema.fields if isinstance(f.dataType, StringType)]

    condition = None
    for c in columns:
        cond = lower(trim(col(c))).isin(NULL_STRINGS)
        sdf = sdf.withColumn(c, when(cond, None).otherwise(col(c)))

    return sdf

StatementMeta(, 7f698acc-f2ee-4717-bc6b-d3383dfadf0e, 19, Finished, Available, Finished, False)

### Cell 3 — Clean Availability Table

In [18]:

def clean_availability_table(source_table: str) -> "DataFrame":
    """
    Apply five data quality transformations to the pre-silver availability table:

    1. Replace null-like string artifacts → actual None
    2. Cast Availability column from string to DoubleType
    3. Scale values in 0–1 range to percentage (multiply by 100)
    4. Round to 2 decimal places
    5. Remove exact duplicate rows

    Also parses the Date column to a proper DateType and standardises
    the Identification column values.
    """
    print(f"🔄 Cleaning availability table: {source_table}")
    sdf = spark.table(source_table)
    raw_count = sdf.count()
    print(f"   Input rows: {raw_count:,}")

    # ── 1. Replace null-like strings ──────────────────────────────────────────
    sdf = replace_null_strings(sdf)

    # ── 2. Cast Availability to numeric ───────────────────────────────────────
    sdf = sdf.withColumn("Availability", col("Availability").cast(DoubleType()))

    # ── 3. Scale 0–1 values to percentage ────────────────────────────────────
    # Values between 0 and 1 (exclusive) represent fractions — multiply by 100
    sdf = sdf.withColumn(
        "Availability",
        when(
            (col("Availability") > 0) & (col("Availability") <= 1),
            col("Availability") * 100
        ).otherwise(col("Availability"))
    )

    # ── 4. Round to 2 decimal places ─────────────────────────────────────────
    sdf = sdf.withColumn("Availability", F.round(col("Availability"), 2))

    # ── 5. Parse Date to DateType ─────────────────────────────────────────────
    sdf = sdf.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))

    # ── 6. Standardise Identification values ─────────────────────────────────
    sdf = sdf.withColumn(
        "FM_Category",
        when(col("Identification") == "Including_FM", "Including FM")
        .when(col("Identification") == "Excluding_FM", "Excluding FM")
        .otherwise(col("Identification"))
    ).drop("Identification")

    # ── 7. Cast ID to integer ─────────────────────────────────────────────────
    sdf = sdf.withColumn("Circuit_ID", col("ID").cast("integer")).drop("ID")

    # ── 8. Remove exact duplicate rows ───────────────────────────────────────
    sdf = sdf.dropDuplicates()

    # ── 9. Add Silver metadata ────────────────────────────────────────────────
    sdf = sdf.withColumn("_silver_transformed_at", F.lit(RUN_TS))

    clean_count   = sdf.count()
    null_avail    = sdf.filter(col("Availability").isNull()).count()
    null_pct      = round((null_avail / clean_count) * 100, 2) if clean_count > 0 else 0
    removed_dupes = raw_count - clean_count

    print(f"   Output rows     : {clean_count:,}")
    print(f"   Duplicates removed: {removed_dupes:,}")
    print(f"   Null Availability : {null_avail:,}  ({null_pct}%)")

    return sdf

StatementMeta(, 7f698acc-f2ee-4717-bc6b-d3383dfadf0e, 20, Finished, Available, Finished, False)

### Cell 4 — Fix Outage Dates (2925 → 2025)

In [19]:

def clean_outage_table(source_table: str) -> "DataFrame":
    """
    Apply data quality transformations to the Bronze outage table:

    1. Replace null-like string artifacts
    2. Parse and fix the Date column (year 2925 → 2025)
    3. Parse Outage Start and Outage End to TimestampType
    4. Cast Outage_in_Secs to numeric
    5. Standardise column names
    6. Remove exact duplicates

    The date fix logic:
        The source data contains fault dates stamped as year 2925 instead of 2025.
        Any date where year > 2100 is clearly wrong for operational telecoms data.
        We subtract 10,800 months (exactly 900 years) using Spark's add_months()
        rather than directly editing the year field — add_months handles leap years,
        month-end dates, and daylight saving edge cases correctly.
    """
    print(f"🔄 Cleaning outage table: {source_table}")
    sdf = spark.table(source_table)
    raw_count = sdf.count()
    print(f"   Input rows: {raw_count:,}")

    # ── 1. Replace null-like strings ──────────────────────────────────────────
    sdf = replace_null_strings(sdf)

    # ── 2. Standardise column names ───────────────────────────────────────────
    rename_map = {
        "Date"                    : "Date",
        "Outage_Start"             : "Outage_Start",
        "Outage_End"               : "Outage_End",
        "Actual_Event_Down_Time"   : "Event_Down_Time",
        "Outage_in_Secs"           : "Outage_Duration_Secs",
        "Type_of_Fault"            : "Fault_Type",
    }
    for old, new in rename_map.items():
        if old in sdf.columns:
            sdf = sdf.withColumnRenamed(old, new)

    # ── 3. Parse and fix the Date column ─────────────────────────────────────
    # Try multiple date formats from source.
    # 'yyyy-MM-dd HH:mm:ss' must come first: Spark 3's strict parser rejects
    # strings like '2025-12-23 00:00:00' when the pattern is 'yyyy-MM-dd'
    # because the trailing time component is treated as unparsed text.
    sdf = sdf.withColumn(
        "Date",
        F.coalesce(
            to_date(col("Date"), "yyyy-MM-dd HH:mm:ss"),
            to_date(col("Date"), "yyyy-MM-dd"),
            to_date(col("Date"), "dd/MM/yyyy"),
            to_date(col("Date"), "MM/dd/yyyy"),
        )
    )

    # Fix year 2925 → 2025 (subtract 900 years = 10,800 months)
    bad_date_count_before = sdf.filter(year(col("Date")) > 2100).count()
    print(f"   Dates with year > 2100 (before fix): {bad_date_count_before}")

    sdf = sdf.withColumn("_year_check", year(col("Date")))
    sdf = sdf.withColumn(
        "Date",
        when(
            col("_year_check") > 2100,
            expr("add_months(Date, -10800)")
        ).otherwise(col("Date"))
    ).drop("_year_check")

    bad_date_count_after = sdf.filter(year(col("Date")) > 2100).count()
    print(f"   Dates with year > 2100 (after fix) : {bad_date_count_after}")

    # ── 4. Apply same fix to Outage_Start and Outage_End timestamps ──────────
    for ts_col in ["Outage_Start", "Outage_End"]:
        if ts_col in sdf.columns:
            sdf = sdf.withColumn(
                ts_col,
                to_date(col(ts_col), "yyyy-MM-dd HH:mm:ss")
            )
            sdf = sdf.withColumn("_year_check", year(col(ts_col)))
            sdf = sdf.withColumn(
                ts_col,
                when(col("_year_check") > 2100,
                     expr(f"add_months({ts_col}, -10800)")
                ).otherwise(col(ts_col))
            ).drop("_year_check")

    # ── 5. Cast Outage_Duration_Secs to numeric ───────────────────────────────
    if "Outage_Duration_Secs" in sdf.columns:
        sdf = sdf.withColumn(
            "Outage_Duration_Secs",
            col("Outage_Duration_Secs").cast(DoubleType())
        )
        sdf = sdf.withColumn(
            "Outage_Duration_Mins",
            F.round(col("Outage_Duration_Secs") / 60, 2)
        )
        sdf = sdf.withColumn(
            "Outage_Duration_Hrs",
            F.round(col("Outage_Duration_Secs") / 3600, 4)
        )

    # ── 6. Cast ID to integer ─────────────────────────────────────────────────
    if "ID" in sdf.columns:
        sdf = sdf.withColumn("Outage_ID", col("ID").cast("integer")).drop("ID")

    # ── 7. Remove exact duplicates ────────────────────────────────────────────
    sdf = sdf.dropDuplicates()

    # ── 8. Add Silver metadata ────────────────────────────────────────────────
    sdf = sdf.withColumn("_silver_transformed_at", F.lit(RUN_TS))

    clean_count = sdf.count()
    print(f"   Output rows: {clean_count:,}")

    return sdf

StatementMeta(, 7f698acc-f2ee-4717-bc6b-d3383dfadf0e, 21, Finished, Available, Finished, False)

### Cell 5 — Run Transformations and Save

In [20]:
# ── Pre-flight check ─────────────────────────────────────────────────────────
# Confirms the pre-silver table exists before attempting transformations.
# If this raises RuntimeError, run nb_PreSilver_StratusCoreTelecoms first.
try:
    spark.table(PRE_SILVER_TABLE).limit(1).count()
    print(f"\u2705 Pre-flight passed: {PRE_SILVER_TABLE} found\n")
except Exception as _e:
    raise RuntimeError(
        f"\n\u274c Pre-flight failed \u2014 {PRE_SILVER_TABLE} not found.\n"
        f"   \u2192 Run nb_PreSilver_StratusCoreTelecoms to completion first.\n"
        f"   Original error: {_e}"
    )

# ── Availability ──────────────────────────────────────────────────────────────
sdf_avail_silver = clean_availability_table(PRE_SILVER_TABLE)

spark.sql("DROP TABLE IF EXISTS silver_stratus_availability")
(
    sdf_avail_silver
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_stratus_availability")
)
print("\u2705 Saved: silver_stratus_availability\n")

# ── Outage ────────────────────────────────────────────────────────────────────
sdf_outage_silver = clean_outage_table(BRONZE_OUTAGE_TABLE)

spark.sql("DROP TABLE IF EXISTS silver_stratus_outage")
(
    sdf_outage_silver
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_stratus_outage")
)
print("\u2705 Saved: silver_stratus_outage")


StatementMeta(, 7f698acc-f2ee-4717-bc6b-d3383dfadf0e, 22, Finished, Available, Finished, False)

✅ Pre-flight passed: lh_Silver_StratusCoreTelecoms.dbo.pre_silver_stratus_availability found

🔄 Cleaning availability table: lh_Silver_StratusCoreTelecoms.dbo.pre_silver_stratus_availability
   Input rows: 4,800
   Output rows     : 4,800
   Duplicates removed: 0
   Null Availability : 0  (0.0%)
✅ Saved: silver_stratus_availability

🔄 Cleaning outage table: lh_Bronze_StratusCoreTelecoms.dbo.bronze_stratus_outage
   Input rows: 200
   Dates with year > 2100 (before fix): 37
   Dates with year > 2100 (after fix) : 0
   Output rows: 200
✅ Saved: silver_stratus_outage


### Cell 6 — Silver Quality Report

In [21]:

print("=" * 60)
print("  SILVER LAYER — DATA QUALITY REPORT")
print("=" * 60)

for table_name, key_col in [
    ("silver_stratus_availability", "Availability"),
    ("silver_stratus_outage",       "Date"),
]:
    sdf   = spark.table(table_name)
    rows  = sdf.count()
    nulls = sdf.filter(col(key_col).isNull()).count()
    null_pct = round((nulls / rows) * 100, 2) if rows > 0 else 0

    print(f"\n📋 {table_name}")
    print(f"   Total rows     : {rows:,}")
    print(f"   Null '{key_col}' : {nulls:,}  ({null_pct}%)")

    if table_name == "silver_stratus_availability":
        stats = sdf.agg(
            F.min("Availability").alias("min"),
            F.max("Availability").alias("max"),
            F.avg("Availability").alias("avg")
        ).collect()[0]
        print(f"   Availability   : min={stats['min']}, max={stats['max']}, avg={round(stats['avg'],2)}")
        print(f"   Date range     : ", end="")
        dates = sdf.agg(F.min("Date"), F.max("Date")).collect()[0]
        print(f"{dates[0]} → {dates[1]}")

    if table_name == "silver_stratus_outage":
        print(f"   Date range     : ", end="")
        dates = sdf.agg(F.min("Date"), F.max("Date")).collect()[0]
        print(f"{dates[0]} → {dates[1]}")
        year_check = sdf.filter(year(col("Date")) > 2100).count()
        print(f"   Dates > 2100   : {year_check}  (should be 0)")

print("\n" + "=" * 60)

StatementMeta(, 7f698acc-f2ee-4717-bc6b-d3383dfadf0e, 23, Finished, Available, Finished, False)

  SILVER LAYER — DATA QUALITY REPORT

📋 silver_stratus_availability
   Total rows     : 4,800
   Null 'Availability' : 0  (0.0%)
   Availability   : min=98.58, max=99.99, avg=99.53
   Date range     : 2025-01-01 → 2025-12-01

📋 silver_stratus_outage
   Total rows     : 200
   Null 'Date' : 0  (0.0%)
   Date range     : 2025-01-04 → 2025-12-31
   Dates > 2100   : 0  (should be 0)

